# 00 — Cliente LLM Robusto (Warm-up de M1)

**Bloom:** Apply | **Duração:** 45 min | **Objetivos:** M1-O3

**Posição:** rodar ANTES do notebook 01. Estabelece o piso técnico (Pydantic + streaming + retry) que o LAB-001 (Agent CLI) presume.

Referência completa: [`labs/lab-000.md`](../labs/lab-000.md).

## Setup

**Provider:** Google Gemini (default da turma — free tier).  
**Como gerar a chave:** ver `API-KEYS.pdf` (mesma pasta) ou https://aistudio.google.com/apikey

**Onde colocar a chave** (a Cell 2 abaixo detecta automaticamente):

| Ambiente | Onde |
|---|---|
| **Google Colab** | 🔑 Secrets (barra lateral) → `GEMINI_API_KEY` → ligar acesso ao notebook |
| **Jupyter local** | arquivo `.env` na raiz da disciplina com `GEMINI_API_KEY=AIza...` |
| **Fallback** | prompt `getpass` (a Cell 2 pede ao rodar, sem persistir) |

Execute as células em ordem (`Run All` funciona end-to-end — nenhuma intervenção manual após informar a chave).

In [2]:
import json
import os
import random
import time
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, Field

# --- Carregar GEMINI_API_KEY conforme o ambiente ---
# Ordem de tentativa: Colab Secrets -> .env local -> getpass prompt.


def _load_gemini_key() -> tuple[str, str]:
    """Retorna (api_key, origem) ou levanta RuntimeError."""
    # 1) Google Colab — Secrets manager
    try:
        from google.colab import userdata  # type: ignore

        try:
            key = userdata.get("GEMINI_API_KEY")
            if key:
                return key, "Colab Secrets"
        except Exception:
            # Secret nao definido ou acesso ao notebook desligado
            pass
    except ImportError:
        pass  # nao esta no Colab

    # 2) Variavel de ambiente / arquivo .env local (Jupyter)
    try:
        from dotenv import find_dotenv, load_dotenv

        dotenv_path = find_dotenv(usecwd=True)
        if dotenv_path:
            load_dotenv(dotenv_path)
    except ImportError:
        pass
    key = os.getenv("GEMINI_API_KEY")
    if key:
        return key, ".env local"

    # 3) Fallback interativo (qualquer ambiente)
    from getpass import getpass

    key = getpass("Cole sua GEMINI_API_KEY (gere em https://aistudio.google.com/apikey): ")
    if not key:
        raise RuntimeError("GEMINI_API_KEY nao fornecida — abortando.")
    return key, "prompt interativo"


api_key, key_source = _load_gemini_key()

# Gemini via endpoint OpenAI-compatible — mesmo SDK, mesma sintaxe.
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "gemini-2.5-flash-lite"

print(f"Provider: Gemini ({MODEL})")
print(f"Key carregada de: {key_source}")

Cole sua GEMINI_API_KEY (gere em https://aistudio.google.com/apikey): ··········
Provider: Gemini (gemini-2.5-flash-lite)
Key carregada de: prompt interativo


## Etapa 1 — Warm-up: 1 call simples

Garantir que a key conecta antes de adicionar complexidade.

In [3]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Diga apenas: ping"}],
    temperature=0.0,
)
print(resp.choices[0].message.content)

ping


**Verificação:** imprime `ping` (ou variação curta). Se erro 401/403 → key inválida; 429 → rate limit (aguardar 60s).

## Etapa 2 — Pydantic Structured Output

Forçar JSON validado em vez de parsear texto livre.

In [4]:
class Classification(BaseModel):
    sentiment: Literal["positivo", "negativo", "neutro"]
    confidence: float = Field(ge=0, le=1)
    reasoning: str = Field(max_length=200)


def classify(text: str) -> Classification:
    schema = Classification.model_json_schema()
    prompt = f"""Classifique o sentimento da mensagem abaixo.
Responda APENAS em JSON valido com este schema:
{json.dumps(schema, indent=2)}

Mensagem: {text}
Output:"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    raw = resp.choices[0].message.content
    return Classification.model_validate_json(raw)


result = classify("Adorei o atendimento, super atencioso!")
print(f"sentiment={result.sentiment}, confidence={result.confidence}")
print(f"reasoning: {result.reasoning}")

sentiment=positivo, confidence=0.95
reasoning: A mensagem expressa claramente satisfação e apreço com o atendimento, utilizando palavras como 'adorei' e 'super atencioso'.


**Reflexão:** o que `model_validate_json()` faz se o LLM retornar `confidence=1.5`?

Resposta: levanta `ValidationError`. Isto é **bom** — detecta drift do modelo cedo, não passa dados inválidos para produção.

## Etapa 3 — Streaming

Streaming reduz TTFT (time-to-first-token). UX percebe ~3× mais rápido.

In [5]:
def stream_completion(prompt: str) -> str:
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    full_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)  # renderiza ao vivo
        full_text += delta
    print()
    return full_text


_ = stream_completion("Liste 5 frameworks Python para LLM, 1 linha cada")

Aqui estão 5 frameworks Python para LLM, cada um em uma linha:

*   **LangChain**: Um framework para desenvolver aplicações com modelos de linguagem.
*   **LlamaIndex**: Um framework para conectar LLMs a dados externos.
*   **Hugging Face Transformers**: Fornece modelos de transformadores pré-treinados para diversas tarefas de PNL.
*   **OpenAI API Client**: Uma biblioteca Python para interagir com a API da OpenAI.
*   **Haystack**: Um framework de código aberto para construir aplicações de busca com LLMs.


**Reflexão:** por que streaming + retry não combina bem?

Se conexão cai no meio do stream, você já mostrou parte ao usuário. Retry gera conteúdo **diferente** (LLM não-determinístico). Estratégias: para chat UI, exibir `[reconectando]`; para batch, não usar streaming.

## Etapa 4 — Retry com Exponential Backoff + Jitter

Production-ready: distinguir erros retryable de non-retryable.

In [6]:
from openai import (
    APIConnectionError,
    APITimeoutError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    RateLimitError,
)

RETRYABLE = (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError)
NON_RETRYABLE = (AuthenticationError, BadRequestError)


def call_with_retry(prompt: str, max_attempts: int = 3) -> str:
    base_delay = 1.0
    last_error: Exception | None = None

    for attempt in range(1, max_attempts + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
            )
            return resp.choices[0].message.content or ""

        except RETRYABLE as e:
            last_error = e
            if attempt == max_attempts:
                break
            delay = base_delay * (2 ** (attempt - 1))
            jitter = random.uniform(0, delay * 0.5)
            wait = delay + jitter
            print(f"  [retry {attempt}] {type(e).__name__}: aguardando {wait:.1f}s")
            time.sleep(wait)

        except NON_RETRYABLE:
            raise

    raise RuntimeError(f"Max retries ({max_attempts}) excedidas") from last_error


result = call_with_retry("Diga: ok")
print(f"Resultado: {result}")

Resultado: Ok


**3 elementos críticos do retry:**

1. Distinção retryable vs não-retryable (auth nunca vira)
2. Exponential backoff (delays 1s, 2s, 4s, 8s)
3. Jitter (random somado ao delay) — evita thundering herd

## Etapa 5 — Pipeline Robusto Combinado

Pydantic + Retry juntos em uma função única.

In [7]:
def robust_classify(text: str, max_attempts: int = 3) -> Classification:
    schema = Classification.model_json_schema()
    prompt = f"""Classifique o sentimento. Responda apenas em JSON:
Schema: {json.dumps(schema)}
Mensagem: {text}
Output:"""

    base_delay = 1.0
    for attempt in range(1, max_attempts + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.0,
            )
            raw = resp.choices[0].message.content or ""
            return Classification.model_validate_json(raw)

        except RETRYABLE as e:
            if attempt == max_attempts:
                raise
            wait = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"  [retry {attempt}] {type(e).__name__}: {wait:.1f}s")
            time.sleep(wait)

        except NON_RETRYABLE:
            raise

    raise RuntimeError("unreachable")


messages_test = [
    "Adorei o atendimento, super atencioso!",
    "Demoraram 3 dias pra responder e nada.",
    "Recebi o produto.",
]

for msg in messages_test:
    out = robust_classify(msg)
    print(f"{msg[:40]:42s} → {out.sentiment} ({out.confidence:.2f})")

Adorei o atendimento, super atencioso!     → positivo (0.95)
Demoraram 3 dias pra responder e nada.     → negativo (0.90)
Recebi o produto.                          → neutro (0.90)


## Verificação

- [ ] Warm-up: 1 call funciona
- [ ] Pydantic: `Classification` valida JSON correto e rejeita inválido
- [ ] Streaming: texto aparece palavra-por-palavra
- [ ] Retry: entende diferença retryable vs non-retryable
- [ ] Pipeline combinado: 3 mensagens classificadas

> **Próximo:** abrir notebook 01 (LAB-001 Agent CLI). As primitivas deste lab reaparecem lá compondo o agente com tool-use.